# DISTCELL2XY feature association and ML-readiness v2

This notebook finds features associated with a configurable large-distance target across the already-audited, one-call-per-row GMLC/IMS/RAW staging tables created by `psap_gmlc_routing_v44.ipynb`.

## Important interpretation

`DISTCELL2XY_M > threshold` is treated here as a **cell-to-reported-location distance anomaly**. It is not automatically proof that a call reached the wrong PSAP. Confirm the source data dictionary before renaming the target to “misroute.” If you also want routed and expected PSAP IDs to differ, set `REQUIRE_ROUTED_EXPECTED_PSAP_MISMATCH = True`.

The correct term is **feature-target association**, not one universal correlation:

- numeric fields: point-biserial correlation, univariate ROC discrimination, and mutual information;
- categorical and network-entity fields: normalized mutual information, corrected Cramér's V, supported/smoothed problem-rate lift;
- free text and SIP/status fields: token/phrase chi-square and regularized log-odds;
- missing fields: missing-versus-present problem-rate difference.

The notebook produces two separate shortlists:

1. **Pre-route prediction** — conservative features available before the routing result.
2. **Post-call detection/diagnosis** — may include IMS, RAW CDDR, SIP messages, alarms, and outcomes.

Associations are screened on the earlier time block only. The optional logistic and Random Forest models are evaluated on the newest held-out block. Random Forest feature impact is measured with held-out **permutation importance**, while its built-in impurity importance is exported only as a secondary diagnostic.


In [ ]:
# ============================ CONFIGURATION ============================
from pathlib import Path

# Existing disk-backed staging database created by the v4 notebook.
DB_PATH = Path(r"C:\temp\gmlc_v2\outputs_psap_rca_v4\psap_rca_stage_v4.sqlite")
OUTPUT_DIR = DB_PATH.parent / "distcell2xy_feature_association_v2"

# Target definition.
DISTANCE_COLUMN = "DISTCELL2XY_M"
DISTANCE_THRESHOLD_M = 20_000.0
REQUIRE_ROUTED_EXPECTED_PSAP_MISMATCH = False
ROUTED_PSAP_COLUMN = "FCC_PSAP_ID"
EXPECTED_PSAP_COLUMN = "EXPECTED_FCC_PSAP_ID"

# Time window. END_DATE_UTC is exclusive when supplied.
# None means: use midnight after the latest parseable call time in the database.
END_DATE_UTC = None                 # example: "2026-08-19"
ANALYSIS_DAYS = 10                  # change to 7, 10, 30, etc.
TIME_COLUMN_OVERRIDE = None         # example: "CALL_BEGIN_TIME_UTC"

# Quality filters.
EXCLUDE_TEST_OR_SIM = True
EXCLUDE_CALL_CONFLICTS = True

# Large-data controls. The full table is scanned narrowly; only this bounded,
# reproducible case/control sample is materialized with every feature.
SQL_CHUNK_ROWS = 100_000
MAX_POSITIVE_CALLS = 20_000
MAX_NEGATIVE_CALLS = 40_000
NEGATIVE_TO_POSITIVE_RATIO = 3
RANDOM_SEED = 42
CACHE_ANALYSIS_SAMPLE = True

# Feature typing and association controls.
TYPE_INFERENCE_ROWS = 10_000
MIN_NON_NULL_ROWS = 100
MIN_CATEGORY_COUNT = 25
MAX_CATEGORY_LEVELS = 300
MIN_ENTITY_CALLS = 25
ENTITY_PRIOR_STRENGTH = 20.0
MAX_TEXT_COLUMNS = 15
TEXT_MIN_DOCUMENTS = 20
TEXT_MAX_TERMS = 15_000
TEXT_TOP_TERMS_PER_COLUMN = 50
TEXT_MAX_CHARS_PER_CALL = 10_000
TOP_FEATURES_PER_SOURCE_AND_TYPE = 25

# Time split. Feature screening uses TRAIN only; validation selects a threshold;
# the newest TEST block is used once for the final estimate.
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15

# Optional baseline model after shortlisting.
RUN_BASELINE_MODEL = True
MODEL_MODE = "POST_CALL_DETECTION"  # or "PRE_ROUTE_PREDICTION"
MAX_MODEL_BASE_FEATURES = 60
INCLUDE_TEXT_IN_BASELINE = True
TARGET_PRECISION = 0.50

# Tree-model confirmation after the association shortlist.
# Keep RUN_BASELINE_MODEL=True because its cell prepares the common train/test frames.
RUN_TREE_MODEL = True
TREE_INCLUDE_TEXT = True
TREE_N_ESTIMATORS = 250
TREE_MAX_DEPTH = 20
TREE_MIN_SAMPLES_LEAF = 10
TREE_MAX_FEATURES = "sqrt"
TREE_MAX_SAMPLES = 0.70
PERMUTATION_IMPORTANCE_REPEATS = 5
PERMUTATION_IMPORTANCE_MAX_ROWS = 10_000
TREE_TOP_FEATURES = 50
TREE_PERMUTATION_MIN_MEAN = 0.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Database:", DB_PATH)
print("Outputs :", OUTPUT_DIR)
print(f"Target  : valid {DISTANCE_COLUMN} > {DISTANCE_THRESHOLD_M:,.0f} m")


In [ ]:
# Imports and reusable helpers.
import json
import math
import re
import sqlite3
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:  # Keeps the notebook testable in a plain Python environment.
    class Markdown(str):
        pass

    def display(value):
        print(value)
from scipy.stats import chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    normalized_mutual_info_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
warnings.filterwarnings("ignore", category=FutureWarning)

MISSING_TOKENS = {"", "NAN", "NONE", "NULL", "<NA>", "<MISSING>"}


def qident(name):
    return '"' + str(name).replace('"', '""') + '"'


def table_columns(connection, table):
    return [row[1] for row in connection.execute(f"PRAGMA table_info({qident(table)})")]


def parse_utc(series):
    """Parse mixed ISO/Oracle-like timestamps as UTC without comparing raw strings."""
    try:
        return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
    except (TypeError, ValueError):
        return pd.to_datetime(series, errors="coerce", utc=True)


def numeric(series):
    return pd.to_numeric(
        series.astype("string").str.replace(",", "", regex=False),
        errors="coerce",
    )


def missing_mask(series):
    as_text = series.astype("string").str.strip()
    return series.isna() | as_text.str.upper().isin(MISSING_TOKENS)


def normalized_id(series):
    result = (
        series.astype("string")
        .str.normalize("NFKC")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
    )
    return result.mask(result.isin(MISSING_TOKENS), "")


def source_and_original(feature):
    for prefix in ("GMLC__", "IMS__", "RAW__", "TIME__", "JOIN__"):
        if feature.startswith(prefix):
            return prefix[:-2], feature[len(prefix):]
    return "DERIVED", feature


def save_csv(frame, filename):
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=False)
    return path


def prevalence(frame):
    if frame.empty:
        return np.nan
    y = frame["__TARGET"].astype(int).to_numpy()
    w = frame["__SAMPLE_WEIGHT"].astype(float).to_numpy()
    return float(np.average(y, weights=w))


def stable_hash(frame):
    text = frame["CALL_KEY"].astype("string") + "|" + str(RANDOM_SEED)
    return pd.util.hash_pandas_object(text, index=False).astype("uint64")


def keep_smallest_hash(current, incoming, limit):
    if incoming.empty:
        return current
    combined = incoming if current is None or current.empty else pd.concat(
        [current, incoming], ignore_index=True
    )
    if limit is None or len(combined) <= limit:
        return combined
    return combined.nsmallest(limit, "__HASH").reset_index(drop=True)


def safe_auc(y, values, weights=None):
    if len(np.unique(y)) < 2:
        return np.nan
    try:
        return float(roc_auc_score(y, values, sample_weight=weights))
    except Exception:
        return np.nan


def normalize_technical_text(value):
    """Retain SIP/status terms while redacting likely subscriber-specific tokens."""
    s = "" if pd.isna(value) else str(value).upper()
    s = s[:TEXT_MAX_CHARS_PER_CALL]
    s = re.sub(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", " <IP> ", s)
    s = re.sub(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+", " <URI> ", s)
    s = re.sub(r"\b[0-9A-F]{12,}\b", " <HEXID> ", s)
    # Preserve three-digit SIP codes such as 401/403/404; redact longer numbers.
    s = re.sub(r"\b\d{4,}\b", " <NUM> ", s)
    return re.sub(r"\s+", " ", s).strip()


In [ ]:
# Validate the existing one-call-per-row staging database and discover schema.
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Staging database not found: {DB_PATH}\n"
        "Run the v4 notebook through the exact GMLC/IMS/RAW enrichment first."
    )

con = sqlite3.connect(str(DB_PATH))
con.execute("PRAGMA query_only=ON")
con.execute("PRAGMA temp_store=FILE")
con.execute("PRAGMA cache_size=-250000")

required_tables = {"gmlc_calls_v4", "ims_agg_v4", "raw_call_agg_v4"}
actual_tables = {
    row[0]
    for row in con.execute("SELECT name FROM sqlite_master WHERE type='table'")
}
missing_tables = required_tables - actual_tables
if missing_tables:
    raise RuntimeError(f"Missing required staging table(s): {sorted(missing_tables)}")

gmlc_cols = table_columns(con, "gmlc_calls_v4")
ims_cols = table_columns(con, "ims_agg_v4")
raw_cols = table_columns(con, "raw_call_agg_v4")

for required in ("CALL_KEY", "GMLC_JOIN_KEY", DISTANCE_COLUMN):
    if required not in gmlc_cols:
        raise RuntimeError(f"gmlc_calls_v4 is missing required column: {required}")

time_candidates = [
    TIME_COLUMN_OVERRIDE,
    "CALL_BEGIN_TIME_UTC",
    "CALL_DATE_UTC",
    "CALL_DATETIME",
    "CALL_DATE",
]
time_candidates = [c for c in time_candidates if c and c in gmlc_cols]
if not time_candidates:
    raise RuntimeError(
        "No supported GMLC time column found. Set TIME_COLUMN_OVERRIDE to an existing column."
    )

# Pick the candidate with the best parse rate on a bounded sample.
time_quality = []
for candidate in time_candidates:
    probe = pd.read_sql_query(
        f"SELECT {qident(candidate)} AS value FROM gmlc_calls_v4 "
        f"WHERE {qident(candidate)} IS NOT NULL LIMIT 10000",
        con,
    )["value"]
    parsed = parse_utc(probe)
    time_quality.append((candidate, float(parsed.notna().mean()) if len(parsed) else 0.0))

TIME_COLUMN = max(time_quality, key=lambda item: item[1])[0]
if max(rate for _, rate in time_quality) < 0.50:
    raise RuntimeError(
        f"Time parsing is too weak: {time_quality}. Set TIME_COLUMN_OVERRIDE explicitly."
    )

schema_summary = pd.DataFrame([
    {"TABLE": "gmlc_calls_v4", "ROWS": con.execute("SELECT COUNT(*) FROM gmlc_calls_v4").fetchone()[0], "COLUMNS": len(gmlc_cols)},
    {"TABLE": "ims_agg_v4", "ROWS": con.execute("SELECT COUNT(*) FROM ims_agg_v4").fetchone()[0], "COLUMNS": len(ims_cols)},
    {"TABLE": "raw_call_agg_v4", "ROWS": con.execute("SELECT COUNT(*) FROM raw_call_agg_v4").fetchone()[0], "COLUMNS": len(raw_cols)},
])
display(schema_summary)
print("Selected time column:", TIME_COLUMN)
print("Time parse probes:", time_quality)


In [ ]:
# Resolve the analysis window. END_DATE_UTC is exclusive; when omitted, the
# notebook uses midnight immediately after the latest parseable call date.
if END_DATE_UTC is None:
    latest_time = None
    for chunk in pd.read_sql_query(
        f"SELECT {qident(TIME_COLUMN)} AS call_time FROM gmlc_calls_v4",
        con,
        chunksize=SQL_CHUNK_ROWS,
    ):
        parsed = parse_utc(chunk["call_time"])
        chunk_max = parsed.max()
        if pd.notna(chunk_max) and (latest_time is None or chunk_max > latest_time):
            latest_time = chunk_max
    if latest_time is None:
        raise RuntimeError(f"No parseable timestamps found in {TIME_COLUMN}")
    window_end = latest_time.floor("D") + pd.Timedelta(days=1)
else:
    window_end = pd.Timestamp(END_DATE_UTC)
    window_end = (
        window_end.tz_localize("UTC")
        if window_end.tzinfo is None
        else window_end.tz_convert("UTC")
    )

window_start = window_end - pd.Timedelta(days=int(ANALYSIS_DAYS))
display(Markdown(
    f"### Analysis window\n"
    f"`[{window_start.isoformat()}, {window_end.isoformat()})`  \n"
    f"Target: valid `{DISTANCE_COLUMN} > {DISTANCE_THRESHOLD_M:,.0f} m`"
))


In [ ]:
# Narrow streaming pass: create the target, count the full population, and keep
# a deterministic bounded case/control sample. Invalid or missing distance is
# UNKNOWN and is never treated as a control.
optional_narrow = [
    "GMLC_ROW_ID",
    ROUTED_PSAP_COLUMN,
    EXPECTED_PSAP_COLUMN,
    "IS_TEST_OR_SIM",
    "CALL_CONFLICT",
]
narrow_cols = list(dict.fromkeys([
    "CALL_KEY",
    "GMLC_JOIN_KEY",
    TIME_COLUMN,
    DISTANCE_COLUMN,
] + [c for c in optional_narrow if c in gmlc_cols]))

if REQUIRE_ROUTED_EXPECTED_PSAP_MISMATCH:
    missing_pair_cols = [
        c for c in (ROUTED_PSAP_COLUMN, EXPECTED_PSAP_COLUMN) if c not in gmlc_cols
    ]
    if missing_pair_cols:
        raise RuntimeError(
            "PSAP-mismatch requirement requested, but these columns are missing: "
            f"{missing_pair_cols}"
        )

select_narrow = ", ".join(qident(c) for c in narrow_cols)
positive_keep = pd.DataFrame()
negative_keep = pd.DataFrame()
full_positive_count = 0
full_negative_count = 0
unknown_distance_count = 0
outside_window_count = 0
excluded_quality_count = 0

for chunk_number, chunk in enumerate(
    pd.read_sql_query(
        f"SELECT {select_narrow} FROM gmlc_calls_v4",
        con,
        chunksize=SQL_CHUNK_ROWS,
    ),
    start=1,
):
    call_time = parse_utc(chunk[TIME_COLUMN])
    in_window = call_time.ge(window_start) & call_time.lt(window_end)
    outside_window_count += int((~in_window).sum())
    chunk = chunk.loc[in_window].copy()
    call_time = call_time.loc[in_window]
    if chunk.empty:
        continue

    quality = pd.Series(True, index=chunk.index)
    if EXCLUDE_TEST_OR_SIM and "IS_TEST_OR_SIM" in chunk:
        quality &= numeric(chunk["IS_TEST_OR_SIM"]).fillna(0).eq(0)
    if EXCLUDE_CALL_CONFLICTS and "CALL_CONFLICT" in chunk:
        quality &= numeric(chunk["CALL_CONFLICT"]).fillna(0).eq(0)
    excluded_quality_count += int((~quality).sum())
    chunk = chunk.loc[quality].copy()
    call_time = call_time.loc[quality]

    distance = numeric(chunk[DISTANCE_COLUMN])
    valid_distance = distance.notna() & np.isfinite(distance) & distance.ge(0)
    unknown_distance_count += int((~valid_distance).sum())
    chunk = chunk.loc[valid_distance].copy()
    call_time = call_time.loc[valid_distance]
    distance = distance.loc[valid_distance]
    if chunk.empty:
        continue

    target = distance.gt(DISTANCE_THRESHOLD_M)
    if REQUIRE_ROUTED_EXPECTED_PSAP_MISMATCH:
        routed = normalized_id(chunk[ROUTED_PSAP_COLUMN])
        expected = normalized_id(chunk[EXPECTED_PSAP_COLUMN])
        valid_pair = routed.ne("") & expected.ne("")
        target &= valid_pair & routed.ne(expected)

    working = pd.DataFrame({
        "CALL_KEY": chunk["CALL_KEY"].astype("string"),
        "GMLC_JOIN_KEY": chunk["GMLC_JOIN_KEY"].astype("string"),
        "__CALL_TIME_UTC": call_time,
        "__DISTANCE_M": distance,
        "__TARGET": target.astype("int8"),
    })
    for c in optional_narrow:
        if c in chunk:
            working[c] = chunk[c]
    working = working[working["CALL_KEY"].notna() & working["CALL_KEY"].ne("")]
    working = working.drop_duplicates("CALL_KEY", keep="first")
    working["__HASH"] = stable_hash(working)

    positives = working[working["__TARGET"].eq(1)]
    negatives = working[working["__TARGET"].eq(0)]
    full_positive_count += len(positives)
    full_negative_count += len(negatives)
    positive_keep = keep_smallest_hash(positive_keep, positives, MAX_POSITIVE_CALLS)
    negative_keep = keep_smallest_hash(negative_keep, negatives, MAX_NEGATIVE_CALLS)

    if chunk_number == 1 or chunk_number % 10 == 0:
        print(
            f"Scanned {chunk_number * SQL_CHUNK_ROWS:,} maximum rows; "
            f"population cases={full_positive_count:,}, controls={full_negative_count:,}"
        )

if full_positive_count == 0:
    raise RuntimeError(
        "No target-positive calls were found. Verify the time window, units, and threshold."
    )
if full_negative_count == 0:
    raise RuntimeError(
        "No controls were found. Keep valid calls at or below the threshold for comparison."
    )

positive_keep = positive_keep.nsmallest(
    min(len(positive_keep), MAX_POSITIVE_CALLS or len(positive_keep)), "__HASH"
)
wanted_controls = min(
    len(negative_keep),
    MAX_NEGATIVE_CALLS or len(negative_keep),
    max(1, int(len(positive_keep) * NEGATIVE_TO_POSITIVE_RATIO)),
)
negative_keep = negative_keep.nsmallest(wanted_controls, "__HASH")

sample_keys = pd.concat([positive_keep, negative_keep], ignore_index=True)
sample_keys = sample_keys.drop(columns="__HASH").drop_duplicates("CALL_KEY")
selected_counts = sample_keys["__TARGET"].value_counts().to_dict()
population_counts = {1: full_positive_count, 0: full_negative_count}
sample_keys["__SAMPLE_WEIGHT"] = sample_keys["__TARGET"].map(
    {
        cls: population_counts[cls] / max(int(selected_counts.get(cls, 0)), 1)
        for cls in (0, 1)
    }
).astype(float)

target_summary = pd.DataFrame([
    {"ITEM": "WINDOW_START_UTC", "VALUE": window_start.isoformat()},
    {"ITEM": "WINDOW_END_UTC_EXCLUSIVE", "VALUE": window_end.isoformat()},
    {"ITEM": "ANALYSIS_DAYS", "VALUE": ANALYSIS_DAYS},
    {"ITEM": "DISTANCE_THRESHOLD_M", "VALUE": DISTANCE_THRESHOLD_M},
    {"ITEM": "REQUIRE_PSAP_MISMATCH", "VALUE": REQUIRE_ROUTED_EXPECTED_PSAP_MISMATCH},
    {"ITEM": "FULL_VALID_PROBLEM_CALLS", "VALUE": full_positive_count},
    {"ITEM": "FULL_VALID_CONTROL_CALLS", "VALUE": full_negative_count},
    {"ITEM": "UNKNOWN_OR_INVALID_DISTANCE_CALLS", "VALUE": unknown_distance_count},
    {"ITEM": "QUALITY_EXCLUDED_CALLS", "VALUE": excluded_quality_count},
    {"ITEM": "SELECTED_PROBLEM_CALLS", "VALUE": int(selected_counts.get(1, 0))},
    {"ITEM": "SELECTED_CONTROL_CALLS", "VALUE": int(selected_counts.get(0, 0))},
])
save_csv(target_summary, "target_summary.csv")
display(target_summary)


In [ ]:
# Materialize every GMLC/IMS/RAW feature only for the selected calls. The v4
# tables are already aggregated to one record per canonical call, avoiding a
# many-to-many join and preventing event-rich calls from receiving extra weight.
con.execute("PRAGMA query_only=OFF")
con.execute("DROP TABLE IF EXISTS temp.association_sample_keys")
con.execute("CREATE TEMP TABLE association_sample_keys (CALL_KEY TEXT PRIMARY KEY)")
con.executemany(
    "INSERT OR IGNORE INTO association_sample_keys(CALL_KEY) VALUES (?)",
    [(str(value),) for value in sample_keys["CALL_KEY"]],
)
con.execute("CREATE INDEX IF NOT EXISTS temp.ix_association_sample_keys ON association_sample_keys(CALL_KEY)")
con.execute("PRAGMA query_only=ON")

select_parts = []
for column in gmlc_cols:
    select_parts.append(f"c.{qident(column)} AS {qident('GMLC__' + column)}")
for column in ims_cols:
    if column != "__KEY":
        select_parts.append(f"i.{qident(column)} AS {qident('IMS__' + column)}")
for column in raw_cols:
    if column != "__KEY":
        select_parts.append(f"r.{qident(column)} AS {qident('RAW__' + column)}")
select_parts.extend([
    "CASE WHEN i.__KEY IS NULL THEN 0 ELSE 1 END AS JOIN__IMS_ATTACHED",
    "CASE WHEN r.__KEY IS NULL THEN 0 ELSE 1 END AS JOIN__RAW_ATTACHED",
])

wide_sql = f"""
SELECT {', '.join(select_parts)}
FROM association_sample_keys s
JOIN gmlc_calls_v4 c ON CAST(c.CALL_KEY AS TEXT)=s.CALL_KEY
LEFT JOIN ims_agg_v4 i ON i.__KEY=c.GMLC_JOIN_KEY
LEFT JOIN raw_call_agg_v4 r ON r.__KEY=c.CALL_KEY
"""

wide_parts = []
for part_number, part in enumerate(
    pd.read_sql_query(wide_sql, con, chunksize=max(5_000, SQL_CHUNK_ROWS // 10)),
    start=1,
):
    wide_parts.append(part)
    print(f"Loaded wide sample part {part_number}: {len(part):,} calls")

analysis_df = pd.concat(wide_parts, ignore_index=True) if wide_parts else pd.DataFrame()
analysis_df["GMLC__CALL_KEY"] = analysis_df["GMLC__CALL_KEY"].astype("string")
analysis_df = analysis_df.merge(
    sample_keys[[
        "CALL_KEY", "__CALL_TIME_UTC", "__DISTANCE_M", "__TARGET", "__SAMPLE_WEIGHT"
    ]],
    left_on="GMLC__CALL_KEY",
    right_on="CALL_KEY",
    how="inner",
    validate="one_to_one",
)
analysis_df = analysis_df.drop(columns="CALL_KEY")

if len(analysis_df) != len(sample_keys):
    raise AssertionError(
        f"Wide join changed call count: selected={len(sample_keys):,}, joined={len(analysis_df):,}"
    )
if analysis_df["GMLC__CALL_KEY"].duplicated().any():
    raise AssertionError("Wide analysis sample is not one row per call")

analysis_df["TIME__HOUR_UTC"] = analysis_df["__CALL_TIME_UTC"].dt.hour.astype("Int16")
analysis_df["TIME__DAY_OF_WEEK_UTC"] = analysis_df["__CALL_TIME_UTC"].dt.dayofweek.astype("Int16")
analysis_df["TIME__IS_WEEKEND_UTC"] = analysis_df["TIME__DAY_OF_WEEK_UTC"].isin([5, 6]).astype("int8")

join_audit = pd.DataFrame([
    {"ITEM": "SELECTED_CALLS", "VALUE": len(analysis_df)},
    {"ITEM": "IMS_ATTACHED", "VALUE": int(numeric(analysis_df["JOIN__IMS_ATTACHED"]).fillna(0).sum())},
    {"ITEM": "RAW_ATTACHED", "VALUE": int(numeric(analysis_df["JOIN__RAW_ATTACHED"]).fillna(0).sum())},
    {"ITEM": "ONE_ROW_PER_CALL", "VALUE": not analysis_df["GMLC__CALL_KEY"].duplicated().any()},
])
save_csv(join_audit, "join_audit.csv")
display(join_audit)

if CACHE_ANALYSIS_SAMPLE:
    analysis_df.to_pickle(OUTPUT_DIR / "analysis_sample.pkl")
print(f"Materialized {len(analysis_df):,} calls x {analysis_df.shape[1]:,} columns")


## Feature inventory and leakage audit

This cell inspects every materialized column and assigns a source, data type, availability stage, and exclusion reason.

Conservative leakage rules remove the target, all distance/geometry/boundary fields, call-level identifiers, routed/expected PSAP outcomes, existing misroute labels, and coordinates used by the distance calculation. IMS/RAW and SIP/failure outcomes remain eligible for **post-call detection**, but not for **pre-route prediction**.


In [ ]:
# Schema-aware feature classification. Review feature_inventory.csv before
# accepting the automatic availability/leakage classification in production.
unique_id_patterns = (
    "CALL_KEY", "UNIQ911_CID", "GMLC_ROW_ID", "RAW_ROW_ID", "SESSION_ID",
    "SESSIONID", "IMSCHARGINGID", "ICID", "CALLID", "CTID", "HDR_TRID",
    "SIP_LINK_ID", "MSISDN", "IMSI", "IMEI", "PHONE", "CALLED_NUMBER",
)
entity_patterns = (
    "ECGI", "CELL", "SECTOR", "ENODEB", "EUTRAN", "SITE_ID", "USID",
    "TAC", "MME", "SBC", "P_CSCF", "PCSCF", "E_CSCF", "ECSCF",
)
text_name_patterns = (
    "STATUS", "REASON", "MESSAGE", "HEADER", "USERAGENT", "USER_AGENT",
    "ROUTE_", "FAILURE", "RESULT", "RESPONSE", "METHOD", "CAUSE",
    "DESCRIPTION", "ERROR", "EXCEPTION", "SIP_", "PCSCF", "ECSCF",
)
direct_leakage_patterns = (
    "DISTCELL", "DISTXY", "DISTANCE_", "BOUNDARY", "GEOMETRY", "LATITUDE",
    "LONGITUDE", "MISROUTE", "ROUTE_INTEGRITY", "EXPECTED_FCC_PSAP",
    "EXPECTED_PSAP", "FCC_PSAP_ID", "PSAP_NAME", "PROBLEM_FLAG",
    "DEFINITE_CORRECT", "DEFINITE_MISROUTE", "CORROBORATION_SCORE",
)
post_route_patterns = (
    "ROUTE_STATUS", "ROUTE_ESINET", "ROUTE_ESZ", "ROUTE_FALLBACK",
    "DEFAULT_ROUTED", "FAILURE", "ANSWERED", "UNAUTHORIZED", "RESPONSE",
    "RESULT", "END_TIME", "DURATION", "COMPLETE_CALL", "LSR_EXCLUDE",
    "SIP_STATUS", "SIP_METHOD", "REGISTER", "CORRELATION_STATUS",
)

inference_source = analysis_df.head(TYPE_INFERENCE_ROWS)
inventory_rows = []

for feature in analysis_df.columns:
    if feature.startswith("__"):
        continue
    source, original = source_and_original(feature)
    upper = original.upper()
    series = inference_source[feature]
    is_missing = missing_mask(series)
    non_missing = series.loc[~is_missing]
    non_null_rows = int((~missing_mask(analysis_df[feature])).sum())
    missing_rate = 1.0 - non_null_rows / max(len(analysis_df), 1)
    n_unique = int(non_missing.astype("string").nunique(dropna=True))
    unique_ratio = n_unique / max(len(non_missing), 1)
    average_length = (
        float(non_missing.astype("string").str.len().mean()) if len(non_missing) else 0.0
    )
    numeric_ratio = 0.0
    if len(non_missing):
        numeric_ratio = float(numeric(non_missing).notna().mean())

    unique_identifier = any(pattern in upper for pattern in unique_id_patterns)
    entity_identifier = any(pattern in upper for pattern in entity_patterns)
    datetime_like = any(token in upper for token in ("DATE", "TIME", "TIMESTAMP"))
    text_named = any(pattern in upper for pattern in text_name_patterns)

    if non_null_rows < MIN_NON_NULL_ROWS:
        inferred_type = "INSUFFICIENT_DATA"
    elif datetime_like and not feature.startswith("TIME__"):
        inferred_type = "DATETIME"
    elif entity_identifier:
        inferred_type = "CATEGORICAL_ENTITY"
    elif unique_identifier:
        inferred_type = "UNIQUE_IDENTIFIER"
    elif numeric_ratio >= 0.95 and n_unique > 2:
        inferred_type = "NUMERIC"
    elif n_unique <= 2:
        inferred_type = "CATEGORICAL"
    elif text_named or (average_length >= 20 and n_unique > MAX_CATEGORY_LEVELS):
        inferred_type = "TEXT"
    elif n_unique <= MAX_CATEGORY_LEVELS or unique_ratio <= 0.20:
        inferred_type = "CATEGORICAL"
    else:
        inferred_type = "HIGH_CARDINALITY"

    leakage_reasons = []
    if original.upper() == DISTANCE_COLUMN.upper():
        leakage_reasons.append("TARGET_SOURCE")
    if any(pattern in upper for pattern in direct_leakage_patterns):
        leakage_reasons.append("TARGET_OR_GEOMETRY_DERIVATIVE")
    if unique_identifier:
        leakage_reasons.append("CALL_LEVEL_IDENTIFIER")
    if feature in {"JOIN__IMS_ATTACHED", "JOIN__RAW_ATTACHED"}:
        # Join coverage can be diagnosed, but should not become a production
        # feature unless the same availability is guaranteed at scoring time.
        leakage_reasons.append("PIPELINE_JOIN_AVAILABILITY")

    if source in {"IMS", "RAW"}:
        availability = "POST_CALL_DIAGNOSTIC"
    elif any(pattern in upper for pattern in post_route_patterns):
        availability = "POST_ROUTE_OR_OUTCOME"
    else:
        availability = "PRE_ROUTE_CANDIDATE"

    inventory_rows.append({
        "SOURCE_TABLE": source,
        "FEATURE": feature,
        "ORIGINAL_COLUMN": original,
        "INFERRED_TYPE": inferred_type,
        "NON_NULL_ROWS": non_null_rows,
        "MISSING_RATE": missing_rate,
        "UNIQUE_VALUES_SAMPLE": n_unique,
        "UNIQUE_RATIO_SAMPLE": unique_ratio,
        "AVERAGE_TEXT_LENGTH_SAMPLE": average_length,
        "NUMERIC_PARSE_RATE_SAMPLE": numeric_ratio,
        "AVAILABLE_STAGE": availability,
        "LEAKAGE_FLAG": bool(leakage_reasons),
        "LEAKAGE_REASON": "|".join(sorted(set(leakage_reasons))),
        "ELIGIBLE_PRE_ROUTE": (
            not leakage_reasons
            and availability == "PRE_ROUTE_CANDIDATE"
            and inferred_type in {"NUMERIC", "CATEGORICAL", "CATEGORICAL_ENTITY", "TEXT"}
        ),
        "ELIGIBLE_POST_CALL_DETECTION": (
            not leakage_reasons
            and inferred_type in {"NUMERIC", "CATEGORICAL", "CATEGORICAL_ENTITY", "TEXT"}
        ),
    })

feature_inventory = pd.DataFrame(inventory_rows).sort_values(
    ["LEAKAGE_FLAG", "SOURCE_TABLE", "INFERRED_TYPE", "FEATURE"]
).reset_index(drop=True)
save_csv(feature_inventory, "feature_inventory.csv")
display(feature_inventory.groupby(
    ["SOURCE_TABLE", "INFERRED_TYPE", "AVAILABLE_STAGE", "LEAKAGE_FLAG"],
    dropna=False,
).size().rename("FEATURES").reset_index())
display(feature_inventory.head(50))


In [ ]:
# Chronological split. Association screening is performed only on the earlier
# TRAIN block, preventing the newest calls from influencing feature selection.
timed = analysis_df[analysis_df["__CALL_TIME_UTC"].notna()].copy()
timed = timed.sort_values(["__CALL_TIME_UTC", "GMLC__CALL_KEY"]).reset_index(drop=True)
if len(timed) < 1_000:
    print("WARNING: fewer than 1,000 timed calls; results will be exploratory.")

train_end_index = max(1, int(len(timed) * TRAIN_FRACTION))
validation_end_index = max(
    train_end_index + 1,
    int(len(timed) * (TRAIN_FRACTION + VALIDATION_FRACTION)),
)
validation_end_index = min(validation_end_index, len(timed) - 1)

train_df = timed.iloc[:train_end_index].copy()
validation_df = timed.iloc[train_end_index:validation_end_index].copy()
test_df = timed.iloc[validation_end_index:].copy()

split_summary = []
for name, frame in (("TRAIN", train_df), ("VALIDATION", validation_df), ("TEST", test_df)):
    split_summary.append({
        "SPLIT": name,
        "ROWS": len(frame),
        "START_UTC": frame["__CALL_TIME_UTC"].min(),
        "END_UTC": frame["__CALL_TIME_UTC"].max(),
        "PROBLEM_ROWS": int(frame["__TARGET"].sum()),
        "WEIGHTED_PROBLEM_RATE": prevalence(frame),
    })
split_summary = pd.DataFrame(split_summary)
save_csv(split_summary, "time_split_summary.csv")
display(split_summary)

for split_name, split_frame in (("TRAIN", train_df), ("VALIDATION", validation_df), ("TEST", test_df)):
    if split_frame["__TARGET"].nunique() < 2:
        raise RuntimeError(
            f"{split_name} contains only one target class. Increase the window/sample caps."
        )


## Association screens

Scores are meaningful **within their feature type**, not as one universal scale. A high score means “useful predictive signal to investigate,” not causation. High-cardinality cell/ECGI/USID results may identify recurring bad entities but can also memorize known infrastructure; they must be tested on future dates and, later, with a held-out-cell evaluation.


In [ ]:
# Numeric features: point-biserial correlation, standardized mean difference,
# univariate AUC discrimination, and mutual information.
eligible_numeric = feature_inventory[
    feature_inventory["INFERRED_TYPE"].eq("NUMERIC")
    & feature_inventory["ELIGIBLE_POST_CALL_DETECTION"]
]["FEATURE"].tolist()

numeric_rows = []
y_all = train_df["__TARGET"].astype(int)
w_all = train_df["__SAMPLE_WEIGHT"].astype(float)

for position, feature in enumerate(eligible_numeric, start=1):
    values = numeric(train_df[feature])
    valid = values.notna() & np.isfinite(values)
    if valid.sum() < MIN_NON_NULL_ROWS or y_all.loc[valid].nunique() < 2:
        continue
    x = values.loc[valid].astype(float)
    y = y_all.loc[valid].astype(int)
    w = w_all.loc[valid].astype(float)
    positive = x[y.eq(1)]
    control = x[y.eq(0)]
    if positive.empty or control.empty or x.nunique() < 2:
        continue

    point_biserial = float(np.corrcoef(x.to_numpy(), y.to_numpy())[0, 1])
    auc = safe_auc(y.to_numpy(), x.to_numpy(), w.to_numpy())
    auc_strength = 2.0 * abs(auc - 0.5) if np.isfinite(auc) else np.nan
    pooled_variance = (positive.var(ddof=1) + control.var(ddof=1)) / 2.0
    standardized_difference = (
        float((positive.mean() - control.mean()) / math.sqrt(pooled_variance))
        if np.isfinite(pooled_variance) and pooled_variance > 0
        else np.nan
    )
    x_for_mi = x.to_numpy().reshape(-1, 1)
    try:
        mi = float(mutual_info_classif(
            x_for_mi,
            y.to_numpy(),
            discrete_features=False,
            random_state=RANDOM_SEED,
        )[0])
    except Exception:
        mi = np.nan

    source, _ = source_and_original(feature)
    numeric_rows.append({
        "SOURCE_TABLE": source,
        "FEATURE": feature,
        "FEATURE_TYPE": "NUMERIC",
        "SUPPORT": int(valid.sum()),
        "MISSING_RATE": float(1.0 - valid.mean()),
        "PROBLEM_MEDIAN": float(positive.median()),
        "CONTROL_MEDIAN": float(control.median()),
        "PROBLEM_MEAN": float(positive.mean()),
        "CONTROL_MEAN": float(control.mean()),
        "POINT_BISERIAL_R": point_biserial,
        "STANDARDIZED_MEAN_DIFFERENCE": standardized_difference,
        "UNIVARIATE_AUC": auc,
        "MUTUAL_INFORMATION": mi,
        "METRIC": "2*ABS(AUC-0.5)",
        "ASSOCIATION_SCORE": auc_strength,
        "EFFECT": standardized_difference,
        "DIRECTION": "HIGHER_IN_PROBLEM" if positive.median() > control.median() else "LOWER_IN_PROBLEM",
        "DETAIL": f"problem median={positive.median():.6g}; control median={control.median():.6g}",
    })
    if position % 25 == 0:
        print(f"Scored {position}/{len(eligible_numeric)} numeric candidates")

numeric_associations = pd.DataFrame(numeric_rows)
if not numeric_associations.empty:
    numeric_associations = numeric_associations.sort_values(
        ["ASSOCIATION_SCORE", "SUPPORT"], ascending=[False, False]
    ).reset_index(drop=True)
save_csv(numeric_associations, "numeric_associations.csv")
display(numeric_associations.head(50))


In [ ]:
# Categorical features and supported network-entity risk.
def corrected_cramers_v(values, target):
    table = pd.crosstab(values, target)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    chi2_value = chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    phi2 = chi2_value / max(n, 1)
    rows, cols = table.shape
    phi2_corrected = max(0.0, phi2 - ((cols - 1) * (rows - 1)) / max(n - 1, 1))
    rows_corrected = rows - ((rows - 1) ** 2) / max(n - 1, 1)
    cols_corrected = cols - ((cols - 1) ** 2) / max(n - 1, 1)
    denominator = min(cols_corrected - 1, rows_corrected - 1)
    return math.sqrt(phi2_corrected / denominator) if denominator > 0 else 0.0


def wilson_lower(successes, total, z=1.96):
    if total <= 0:
        return np.nan
    p = successes / total
    denominator = 1 + z * z / total
    centre = p + z * z / (2 * total)
    margin = z * math.sqrt((p * (1 - p) + z * z / (4 * total)) / total)
    return (centre - margin) / denominator


eligible_categorical = feature_inventory[
    feature_inventory["INFERRED_TYPE"].isin(["CATEGORICAL", "CATEGORICAL_ENTITY"])
    & feature_inventory["ELIGIBLE_POST_CALL_DETECTION"]
]["FEATURE"].tolist()

categorical_feature_rows = []
categorical_level_rows = []
entity_level_rows = []
weighted_base_rate = prevalence(train_df)

for position, feature in enumerate(eligible_categorical, start=1):
    raw_values = train_df[feature].astype("string").str.strip()
    values = raw_values.mask(missing_mask(train_df[feature]), "<MISSING>")
    counts = values.value_counts(dropna=False)
    collapsed = values.where(values.map(counts).ge(MIN_CATEGORY_COUNT), "<RARE>")
    if collapsed.nunique() < 2:
        continue
    y = train_df["__TARGET"].astype(int)
    w = train_df["__SAMPLE_WEIGHT"].astype(float)
    nmi = float(normalized_mutual_info_score(y, collapsed))
    cramer = float(corrected_cramers_v(collapsed, y))
    source, original = source_and_original(feature)

    level_frame = pd.DataFrame({"LEVEL": values, "Y": y, "W": w})
    level_frame["WY"] = level_frame["W"] * level_frame["Y"]
    grouped = level_frame.groupby("LEVEL", dropna=False).agg(
        SAMPLE_CALLS=("Y", "size"),
        SAMPLE_PROBLEM_CALLS=("Y", "sum"),
        POPULATION_WEIGHT=("W", "sum"),
        WEIGHTED_PROBLEM_CALLS=("WY", "sum"),
    ).reset_index()
    grouped["WEIGHTED_PROBLEM_RATE"] = (
        grouped["WEIGHTED_PROBLEM_CALLS"] / grouped["POPULATION_WEIGHT"].replace(0, np.nan)
    )
    grouped["SMOOTHED_PROBLEM_RATE"] = (
        grouped["SAMPLE_PROBLEM_CALLS"] + ENTITY_PRIOR_STRENGTH * weighted_base_rate
    ) / (grouped["SAMPLE_CALLS"] + ENTITY_PRIOR_STRENGTH)
    grouped["LIFT_VS_GLOBAL"] = grouped["SMOOTHED_PROBLEM_RATE"] / max(weighted_base_rate, 1e-12)
    grouped["WILSON_LOWER_95"] = [
        wilson_lower(int(bad), int(total))
        for bad, total in zip(grouped["SAMPLE_PROBLEM_CALLS"], grouped["SAMPLE_CALLS"])
    ]
    grouped.insert(0, "FEATURE", feature)
    grouped.insert(0, "SOURCE_TABLE", source)
    categorical_level_rows.append(grouped)

    supported = grouped[
        grouped["SAMPLE_CALLS"].ge(MIN_CATEGORY_COUNT)
        & grouped["LEVEL"].ne("<MISSING>")
    ]
    strongest = (
        supported.sort_values(
            ["WILSON_LOWER_95", "SAMPLE_CALLS"], ascending=[False, False]
        ).iloc[0]
        if not supported.empty
        else None
    )
    categorical_feature_rows.append({
        "SOURCE_TABLE": source,
        "FEATURE": feature,
        "FEATURE_TYPE": "CATEGORICAL_ENTITY" if any(p in original.upper() for p in entity_patterns) else "CATEGORICAL",
        "SUPPORT": int(len(values)),
        "MISSING_RATE": float(values.eq("<MISSING>").mean()),
        "LEVELS_AFTER_COLLAPSE": int(collapsed.nunique()),
        "NORMALIZED_MUTUAL_INFORMATION": nmi,
        "CORRECTED_CRAMERS_V": cramer,
        "STRONGEST_SUPPORTED_LEVEL": None if strongest is None else strongest["LEVEL"],
        "STRONGEST_LEVEL_CALLS": None if strongest is None else int(strongest["SAMPLE_CALLS"]),
        "STRONGEST_LEVEL_LIFT": None if strongest is None else float(strongest["LIFT_VS_GLOBAL"]),
        "METRIC": "NORMALIZED_MUTUAL_INFORMATION",
        "ASSOCIATION_SCORE": nmi,
        "EFFECT": None if strongest is None else float(strongest["LIFT_VS_GLOBAL"]),
        "DIRECTION": "LEVEL_SPECIFIC_RISK",
        "DETAIL": (
            "No supported level"
            if strongest is None
            else f"top supported level={strongest['LEVEL']}; lift={strongest['LIFT_VS_GLOBAL']:.3f}"
        ),
    })

    inventory_type = feature_inventory.set_index("FEATURE").at[feature, "INFERRED_TYPE"]
    if inventory_type == "CATEGORICAL_ENTITY":
        entity_supported = grouped[
            grouped["SAMPLE_CALLS"].ge(MIN_ENTITY_CALLS)
        ].copy()
        if not entity_supported.empty:
            entity_supported["ACTIVE_DAYS_TRAIN"] = (
                train_df.assign(
                    __ENTITY_LEVEL=values,
                    __DAY=train_df["__CALL_TIME_UTC"].dt.floor("D"),
                )
                .groupby("__ENTITY_LEVEL")["__DAY"]
                .nunique()
                .reindex(entity_supported["LEVEL"])
                .to_numpy()
            )
            entity_level_rows.append(entity_supported)

    if position % 25 == 0:
        print(f"Scored {position}/{len(eligible_categorical)} categorical candidates")

categorical_associations = pd.DataFrame(categorical_feature_rows)
if not categorical_associations.empty:
    categorical_associations = categorical_associations.sort_values(
        ["ASSOCIATION_SCORE", "SUPPORT"], ascending=[False, False]
    ).reset_index(drop=True)
categorical_value_associations = (
    pd.concat(categorical_level_rows, ignore_index=True)
    if categorical_level_rows else pd.DataFrame()
)
entity_risk = (
    pd.concat(entity_level_rows, ignore_index=True)
    if entity_level_rows else pd.DataFrame()
)
if not entity_risk.empty:
    entity_risk = entity_risk.sort_values(
        ["WILSON_LOWER_95", "LIFT_VS_GLOBAL", "SAMPLE_CALLS"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

save_csv(categorical_associations, "categorical_feature_associations.csv")
save_csv(categorical_value_associations, "categorical_value_associations.csv")
save_csv(entity_risk, "network_entity_risk.csv")
display(categorical_associations.head(50))
display(Markdown("### Supported ECGI/cell/USID/network-entity risk"))
display(entity_risk.head(75))


In [ ]:
# Missingness is scored independently. A high signal can be operationally useful,
# but join-caused missingness should be treated as pipeline quality, not network cause.
eligible_detection_features = feature_inventory[
    feature_inventory["ELIGIBLE_POST_CALL_DETECTION"]
]["FEATURE"].tolist()

missing_rows = []
weighted_global_rate = prevalence(train_df)
for feature in eligible_detection_features:
    missing = missing_mask(train_df[feature])
    if missing.all() or (~missing).all():
        continue
    y = train_df["__TARGET"].astype(int)
    w = train_df["__SAMPLE_WEIGHT"].astype(float)
    missing_rate = float(np.average(y[missing], weights=w[missing])) if missing.any() else np.nan
    present_rate = float(np.average(y[~missing], weights=w[~missing])) if (~missing).any() else np.nan
    difference = missing_rate - present_rate
    source, _ = source_and_original(feature)
    missing_rows.append({
        "SOURCE_TABLE": source,
        "FEATURE": feature,
        "FEATURE_TYPE": "MISSINGNESS",
        "SUPPORT": int(len(train_df)),
        "MISSING_ROWS": int(missing.sum()),
        "MISSING_RATE": float(missing.mean()),
        "PROBLEM_RATE_WHEN_MISSING": missing_rate,
        "PROBLEM_RATE_WHEN_PRESENT": present_rate,
        "LIFT_WHEN_MISSING": missing_rate / max(weighted_global_rate, 1e-12),
        "METRIC": "ABS(MISSING_RATE-PRESENT_RATE)",
        "ASSOCIATION_SCORE": abs(difference),
        "EFFECT": difference,
        "DIRECTION": "MISSING_HIGHER_RISK" if difference > 0 else "PRESENT_HIGHER_RISK",
        "DETAIL": f"missing rate={missing_rate:.6f}; present rate={present_rate:.6f}",
    })

missingness_associations = pd.DataFrame(missing_rows)
if not missingness_associations.empty:
    missingness_associations = missingness_associations.sort_values(
        ["ASSOCIATION_SCORE", "MISSING_ROWS"], ascending=[False, False]
    ).reset_index(drop=True)
save_csv(missingness_associations, "missingness_associations.csv")
display(missingness_associations.head(50))


In [ ]:
# SIP/message/free-text token screening. Each column is analyzed separately so
# every token retains its source table and source column. Long numbers, IPs,
# subscriber-like IDs, and URI user parts are redacted in the exported report.
text_candidates = feature_inventory[
    feature_inventory["INFERRED_TYPE"].eq("TEXT")
    & feature_inventory["ELIGIBLE_POST_CALL_DETECTION"]
].sort_values(
    ["NON_NULL_ROWS", "AVERAGE_TEXT_LENGTH_SAMPLE"], ascending=[False, False]
).head(MAX_TEXT_COLUMNS)

text_token_frames = []
text_feature_rows = []
y_text = train_df["__TARGET"].astype(int).to_numpy()

for position, row in enumerate(text_candidates.itertuples(index=False), start=1):
    feature = row.FEATURE
    documents = train_df[feature].map(normalize_technical_text)
    if documents.str.len().gt(0).sum() < MIN_NON_NULL_ROWS:
        continue
    vectorizer = TfidfVectorizer(
        lowercase=False,
        binary=True,
        use_idf=False,
        norm=None,
        min_df=TEXT_MIN_DOCUMENTS,
        max_features=TEXT_MAX_TERMS,
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b[\w][\w:/._<>-]{1,}\b",
    )
    try:
        matrix = vectorizer.fit_transform(documents)
    except ValueError as exc:
        print(f"Skipped text feature {feature}: {exc}")
        continue
    if matrix.shape[1] == 0:
        continue

    chi_values, p_values = chi2(matrix, y_text)
    terms = vectorizer.get_feature_names_out()
    positive_mask = y_text == 1
    control_mask = y_text == 0
    positive_support = np.asarray((matrix[positive_mask] > 0).sum(axis=0)).ravel()
    control_support = np.asarray((matrix[control_mask] > 0).sum(axis=0)).ravel()
    positive_total = max(int(positive_mask.sum()), 1)
    control_total = max(int(control_mask.sum()), 1)
    alpha = 0.5
    positive_rate = (positive_support + alpha) / (positive_total + 2 * alpha)
    control_rate = (control_support + alpha) / (control_total + 2 * alpha)
    log_odds = np.log(positive_rate / (1 - positive_rate)) - np.log(
        control_rate / (1 - control_rate)
    )

    token_frame = pd.DataFrame({
        "SOURCE_TABLE": row.SOURCE_TABLE,
        "FEATURE": feature,
        "TOKEN_OR_PHRASE": terms,
        "CHI2": chi_values,
        "P_VALUE_UNADJUSTED": p_values,
        "PROBLEM_DOCUMENTS": positive_support,
        "CONTROL_DOCUMENTS": control_support,
        "PROBLEM_DOCUMENT_RATE": positive_support / positive_total,
        "CONTROL_DOCUMENT_RATE": control_support / control_total,
        "LOG_ODDS_PROBLEM_VS_CONTROL": log_odds,
    })
    token_frame["TOTAL_DOCUMENTS_WITH_TOKEN"] = (
        token_frame["PROBLEM_DOCUMENTS"] + token_frame["CONTROL_DOCUMENTS"]
    )
    token_frame = token_frame.sort_values(
        ["CHI2", "TOTAL_DOCUMENTS_WITH_TOKEN"], ascending=[False, False]
    ).head(TEXT_TOP_TERMS_PER_COLUMN)
    text_token_frames.append(token_frame)

    strongest = token_frame.iloc[0]
    transformed_score = 1.0 - math.exp(-abs(float(strongest["LOG_ODDS_PROBLEM_VS_CONTROL"])) / 2.0)
    text_feature_rows.append({
        "SOURCE_TABLE": row.SOURCE_TABLE,
        "FEATURE": feature,
        "FEATURE_TYPE": "TEXT",
        "SUPPORT": int(documents.str.len().gt(0).sum()),
        "MISSING_RATE": float(documents.str.len().eq(0).mean()),
        "TOP_TOKEN_OR_PHRASE": strongest["TOKEN_OR_PHRASE"],
        "TOP_TOKEN_CHI2": float(strongest["CHI2"]),
        "TOP_TOKEN_LOG_ODDS": float(strongest["LOG_ODDS_PROBLEM_VS_CONTROL"]),
        "METRIC": "TRANSFORMED_ABS(TOP_TOKEN_LOG_ODDS)",
        "ASSOCIATION_SCORE": transformed_score,
        "EFFECT": float(strongest["LOG_ODDS_PROBLEM_VS_CONTROL"]),
        "DIRECTION": (
            "TOKEN_MORE_COMMON_IN_PROBLEM"
            if strongest["LOG_ODDS_PROBLEM_VS_CONTROL"] > 0
            else "TOKEN_MORE_COMMON_IN_CONTROL"
        ),
        "DETAIL": f"top token/phrase={strongest['TOKEN_OR_PHRASE']}; chi2={strongest['CHI2']:.3f}",
    })
    print(f"Text {position}/{len(text_candidates)}: {feature} ({matrix.shape[1]:,} terms)")

text_token_associations = (
    pd.concat(text_token_frames, ignore_index=True)
    if text_token_frames else pd.DataFrame()
)
text_feature_associations = pd.DataFrame(text_feature_rows)
if not text_feature_associations.empty:
    text_feature_associations = text_feature_associations.sort_values(
        ["ASSOCIATION_SCORE", "SUPPORT"], ascending=[False, False]
    ).reset_index(drop=True)

save_csv(text_token_associations, "text_token_associations.csv")
save_csv(text_feature_associations, "text_feature_associations.csv")
display(text_feature_associations)
display(text_token_associations.head(100))


In [ ]:
# Build one association catalog and two leakage-aware ML candidate shortlists.
common_columns = [
    "SOURCE_TABLE", "FEATURE", "FEATURE_TYPE", "SUPPORT", "MISSING_RATE",
    "METRIC", "ASSOCIATION_SCORE", "EFFECT", "DIRECTION", "DETAIL",
]
association_parts = []
for frame in (
    numeric_associations,
    categorical_associations,
    missingness_associations,
    text_feature_associations,
):
    if frame is not None and not frame.empty:
        prepared = frame.copy()
        for column in common_columns:
            if column not in prepared:
                prepared[column] = np.nan
        association_parts.append(prepared[common_columns])

feature_association_master = (
    pd.concat(association_parts, ignore_index=True)
    if association_parts else pd.DataFrame(columns=common_columns)
)

inventory_lookup = feature_inventory.set_index("FEATURE")
feature_association_master["AVAILABLE_STAGE"] = feature_association_master["FEATURE"].map(
    inventory_lookup["AVAILABLE_STAGE"]
)
feature_association_master["LEAKAGE_FLAG"] = feature_association_master["FEATURE"].map(
    inventory_lookup["LEAKAGE_FLAG"]
).fillna(False)
feature_association_master["LEAKAGE_REASON"] = feature_association_master["FEATURE"].map(
    inventory_lookup["LEAKAGE_REASON"]
).fillna("")
feature_association_master["ELIGIBLE_PRE_ROUTE"] = feature_association_master["FEATURE"].map(
    inventory_lookup["ELIGIBLE_PRE_ROUTE"]
).fillna(False)
feature_association_master["ELIGIBLE_POST_CALL_DETECTION"] = feature_association_master["FEATURE"].map(
    inventory_lookup["ELIGIBLE_POST_CALL_DETECTION"]
).fillna(False)

feature_association_master["RANK_WITHIN_SOURCE_AND_TYPE"] = (
    feature_association_master.groupby(["SOURCE_TABLE", "FEATURE_TYPE"])["ASSOCIATION_SCORE"]
    .rank(method="first", ascending=False)
    .astype("Int64")
)
feature_association_master = feature_association_master.sort_values(
    ["SOURCE_TABLE", "FEATURE_TYPE", "RANK_WITHIN_SOURCE_AND_TYPE"]
).reset_index(drop=True)

minimum_support = max(MIN_NON_NULL_ROWS, MIN_CATEGORY_COUNT)
base_filter = (
    ~feature_association_master["LEAKAGE_FLAG"]
    & feature_association_master["ELIGIBLE_POST_CALL_DETECTION"]
    & feature_association_master["ASSOCIATION_SCORE"].notna()
    & feature_association_master["SUPPORT"].ge(minimum_support)
    & feature_association_master["RANK_WITHIN_SOURCE_AND_TYPE"].le(
        TOP_FEATURES_PER_SOURCE_AND_TYPE
    )
)

feature_shortlist_detection = feature_association_master[base_filter].copy()
feature_shortlist_prediction = feature_association_master[
    base_filter & feature_association_master["ELIGIBLE_PRE_ROUTE"]
].copy()

# Within-type ranking is deliberate: AUC strength, NMI, missing-rate difference,
# and text log-odds are not a single interchangeable measurement scale.
save_csv(feature_association_master, "feature_association_master.csv")
save_csv(feature_shortlist_prediction, "feature_shortlist_pre_route_prediction.csv")
save_csv(feature_shortlist_detection, "feature_shortlist_post_call_detection.csv")

display(Markdown("### Pre-route prediction shortlist"))
display(feature_shortlist_prediction.head(100))
display(Markdown("### Post-call detection/diagnosis shortlist"))
display(feature_shortlist_detection.head(150))


In [ ]:
# Compact visualization of the strongest within-type signals.
import matplotlib.pyplot as plt

plot_frame = feature_shortlist_detection.copy()
plot_frame = plot_frame.sort_values(
    ["SOURCE_TABLE", "FEATURE_TYPE", "ASSOCIATION_SCORE"],
    ascending=[True, True, False],
).groupby(["SOURCE_TABLE", "FEATURE_TYPE"], group_keys=False).head(8)

if not plot_frame.empty:
    groups = list(plot_frame.groupby(["SOURCE_TABLE", "FEATURE_TYPE"]))
    fig, axes = plt.subplots(
        len(groups), 1, figsize=(13, max(3.5, 3.0 * len(groups))), squeeze=False
    )
    for axis, ((source, feature_type), group) in zip(axes.ravel(), groups):
        group = group.sort_values("ASSOCIATION_SCORE")
        labels = [name.replace(source + "__", "")[-55:] for name in group["FEATURE"]]
        axis.barh(labels, group["ASSOCIATION_SCORE"], color="#276FBF")
        axis.set_title(f"{source} — {feature_type} (rank only within this type)")
        axis.set_xlabel("Association score")
        axis.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plot_path = OUTPUT_DIR / "top_associations_by_source_and_type.png"
    plt.savefig(plot_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", plot_path)


## Optional chronological baseline

This is an interpretable first model, not the final production model. It uses only shortlisted fields, fits on the earlier block, chooses an operating threshold on validation data, and evaluates once on the newest block.

- `MODEL_MODE = "PRE_ROUTE_PREDICTION"` excludes IMS/RAW and route outcomes.
- `MODEL_MODE = "POST_CALL_DETECTION"` may use SIP/status/alarm evidence after the routing event.

For a later production study, also run a held-out-cell/ECGI evaluation. Otherwise an ID-heavy model may simply memorize previously problematic cells.


In [ ]:
# Optional regularized logistic baseline with chronological evaluation.
if RUN_BASELINE_MODEL:
    from joblib import dump

    if MODEL_MODE == "PRE_ROUTE_PREDICTION":
        active_shortlist = feature_shortlist_prediction.copy()
    elif MODEL_MODE == "POST_CALL_DETECTION":
        active_shortlist = feature_shortlist_detection.copy()
    else:
        raise ValueError("MODEL_MODE must be PRE_ROUTE_PREDICTION or POST_CALL_DETECTION")

    active_shortlist = active_shortlist.sort_values(
        ["RANK_WITHIN_SOURCE_AND_TYPE", "ASSOCIATION_SCORE"],
        ascending=[True, False],
    )

    base_rows = active_shortlist[
        active_shortlist["FEATURE_TYPE"].isin(
            ["NUMERIC", "CATEGORICAL", "CATEGORICAL_ENTITY", "TEXT"]
        )
    ]
    selected_features = list(dict.fromkeys(base_rows["FEATURE"].tolist()))[:MAX_MODEL_BASE_FEATURES]
    selected_missing = list(dict.fromkeys(
        active_shortlist.loc[
            active_shortlist["FEATURE_TYPE"].eq("MISSINGNESS"), "FEATURE"
        ].tolist()
    ))[:20]

    selected_features = [f for f in selected_features if f in analysis_df.columns]
    selected_missing = [f for f in selected_missing if f in analysis_df.columns]
    if not selected_features and not selected_missing:
        raise RuntimeError("No shortlisted features are available for the baseline model")

    type_lookup = feature_inventory.set_index("FEATURE")["INFERRED_TYPE"].to_dict()
    numeric_features = [f for f in selected_features if type_lookup.get(f) == "NUMERIC"]
    categorical_features = [
        f for f in selected_features
        if type_lookup.get(f) in {"CATEGORICAL", "CATEGORICAL_ENTITY"}
    ]
    text_features = [f for f in selected_features if type_lookup.get(f) == "TEXT"]
    if not INCLUDE_TEXT_IN_BASELINE:
        text_features = []

    def build_model_frame(frame):
        result = pd.DataFrame(index=frame.index)
        for feature in numeric_features:
            result[feature] = numeric(frame[feature])
        for feature in categorical_features:
            result[feature] = frame[feature].astype("string").mask(
                missing_mask(frame[feature]), "<MISSING>"
            )
        for feature in selected_missing:
            result[f"MISSING__{feature}"] = missing_mask(frame[feature]).astype("int8")
        if text_features:
            text_parts = []
            for feature in text_features:
                prefix = source_and_original(feature)[1] + "="
                text_parts.append(prefix + frame[feature].map(normalize_technical_text))
            combined = text_parts[0]
            for part in text_parts[1:]:
                combined = combined + " | " + part
            result["__MODEL_TEXT"] = combined
        return result

    X_train = build_model_frame(train_df)
    X_validation = build_model_frame(validation_df)
    X_test = build_model_frame(test_df)
    y_train = train_df["__TARGET"].astype(int)
    y_validation = validation_df["__TARGET"].astype(int)
    y_test = test_df["__TARGET"].astype(int)
    w_train = train_df["__SAMPLE_WEIGHT"].astype(float)
    w_validation = validation_df["__SAMPLE_WEIGHT"].astype(float)
    w_test = test_df["__SAMPLE_WEIGHT"].astype(float)

    transformers = []
    model_numeric = numeric_features + [f"MISSING__{f}" for f in selected_missing]
    if model_numeric:
        transformers.append((
            "numeric",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler(with_mean=False)),
            ]),
            model_numeric,
        ))
    if categorical_features:
        transformers.append((
            "categorical",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="infrequent_if_exist",
                    min_frequency=MIN_CATEGORY_COUNT,
                )),
            ]),
            categorical_features,
        ))
    if text_features:
        transformers.append((
            "text",
            TfidfVectorizer(
                lowercase=False,
                min_df=TEXT_MIN_DOCUMENTS,
                max_features=TEXT_MAX_TERMS,
                ngram_range=(1, 2),
                token_pattern=r"(?u)\b[\w][\w:/._<>-]{1,}\b",
                sublinear_tf=True,
            ),
            "__MODEL_TEXT",
        ))

    pipeline = Pipeline([
        ("features", ColumnTransformer(transformers, remainder="drop")),
        ("model", LogisticRegression(
            solver="saga",
            penalty="l2",
            class_weight="balanced",
            max_iter=500,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )),
    ])
    pipeline.fit(X_train, y_train, model__sample_weight=w_train)

    validation_probability = pipeline.predict_proba(X_validation)[:, 1]
    precision_curve, recall_curve, thresholds = precision_recall_curve(
        y_validation,
        validation_probability,
        sample_weight=w_validation,
    )
    eligible_thresholds = np.where(precision_curve[:-1] >= TARGET_PRECISION)[0]
    if len(eligible_thresholds):
        best_index = eligible_thresholds[np.argmax(recall_curve[:-1][eligible_thresholds])]
    else:
        f1_curve = 2 * precision_curve[:-1] * recall_curve[:-1] / np.maximum(
            precision_curve[:-1] + recall_curve[:-1], 1e-12
        )
        best_index = int(np.nanargmax(f1_curve))
    operating_threshold = float(thresholds[best_index])

    test_probability = pipeline.predict_proba(X_test)[:, 1]
    test_prediction = (test_probability >= operating_threshold).astype(int)
    top_k = max(1, int(math.ceil(0.05 * len(test_probability))))
    top_indices = np.argsort(-test_probability)[:top_k]
    recall_at_top_5pct = float(
        y_test.iloc[top_indices].sum() / max(int(y_test.sum()), 1)
    )
    false_positives = int(((test_prediction == 1) & (y_test.to_numpy() == 0)).sum())

    baseline_metrics = {
        "model_mode": MODEL_MODE,
        "distance_threshold_m": DISTANCE_THRESHOLD_M,
        "operating_threshold": operating_threshold,
        "target_precision_requested": TARGET_PRECISION,
        "test_rows_sample": int(len(y_test)),
        "test_problem_rows_sample": int(y_test.sum()),
        "test_pr_auc_weighted": float(average_precision_score(
            y_test, test_probability, sample_weight=w_test
        )),
        "test_roc_auc_weighted": float(roc_auc_score(
            y_test, test_probability, sample_weight=w_test
        )),
        "test_precision_weighted": float(precision_score(
            y_test, test_prediction, sample_weight=w_test, zero_division=0
        )),
        "test_recall_weighted": float(recall_score(
            y_test, test_prediction, sample_weight=w_test, zero_division=0
        )),
        "test_f1_weighted": float(f1_score(
            y_test, test_prediction, sample_weight=w_test, zero_division=0
        )),
        "sample_false_alarms_per_1000": 1000.0 * false_positives / max(len(y_test), 1),
        "sample_recall_at_top_5_percent": recall_at_top_5pct,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "text_features": text_features,
        "missingness_features": selected_missing,
    }

    model_path = OUTPUT_DIR / f"baseline_{MODEL_MODE.lower()}.joblib"
    metrics_path = OUTPUT_DIR / f"baseline_{MODEL_MODE.lower()}_metrics.json"
    dump(pipeline, model_path)
    metrics_path.write_text(json.dumps(baseline_metrics, indent=2), encoding="utf-8")
    display(pd.DataFrame([
        {"METRIC": key, "VALUE": value}
        for key, value in baseline_metrics.items()
        if not isinstance(value, list)
    ]))
    print("Saved model  :", model_path)
    print("Saved metrics:", metrics_path)
else:
    print("RUN_BASELINE_MODEL=False; association reports and shortlists are complete.")


## Random Forest confirmation and model-based feature impact

The association tables above are univariate: they identify fields related to the target one at a time. This cell adds a multivariate tree model to test nonlinearities and interactions.

The primary model-based ranking is **held-out permutation importance**: each original model input is shuffled on the newest test block and the loss of PR-AUC is measured. This is preferred over the Random Forest's built-in impurity importance, which can overvalue IDs and high-cardinality fields. Built-in impurity importance is still exported separately for diagnosis.

Permutation importance means predictive contribution within this fitted model, not causality. Revalidate high-ranking ECGI/cell/USID features on future dates and held-out network entities.


In [ ]:
# Optional Random Forest using the same leakage-safe shortlist and chronological split.
if RUN_TREE_MODEL:
    if not RUN_BASELINE_MODEL:
        raise RuntimeError(
            "RUN_TREE_MODEL=True requires RUN_BASELINE_MODEL=True so the common "
            "shortlist and chronological model frames are prepared first."
        )

    tree_transformers = []
    for transformer_name, transformer, transformer_columns in transformers:
        if transformer_name == "text" and not TREE_INCLUDE_TEXT:
            continue
        tree_transformers.append((
            transformer_name,
            clone(transformer),
            transformer_columns,
        ))

    tree_pipeline = Pipeline([
        ("features", ColumnTransformer(tree_transformers, remainder="drop")),
        ("model", RandomForestClassifier(
            n_estimators=TREE_N_ESTIMATORS,
            max_depth=TREE_MAX_DEPTH,
            min_samples_leaf=TREE_MIN_SAMPLES_LEAF,
            max_features=TREE_MAX_FEATURES,
            max_samples=TREE_MAX_SAMPLES,
            bootstrap=True,
            class_weight="balanced_subsample",
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )),
    ])
    tree_pipeline.fit(X_train, y_train, model__sample_weight=w_train)

    tree_validation_probability = tree_pipeline.predict_proba(X_validation)[:, 1]
    tree_precision_curve, tree_recall_curve, tree_thresholds = precision_recall_curve(
        y_validation,
        tree_validation_probability,
        sample_weight=w_validation,
    )
    tree_eligible_thresholds = np.where(
        tree_precision_curve[:-1] >= TARGET_PRECISION
    )[0]
    if len(tree_eligible_thresholds):
        tree_best_index = tree_eligible_thresholds[
            np.argmax(tree_recall_curve[:-1][tree_eligible_thresholds])
        ]
    else:
        tree_f1_curve = (
            2 * tree_precision_curve[:-1] * tree_recall_curve[:-1]
            / np.maximum(
                tree_precision_curve[:-1] + tree_recall_curve[:-1],
                1e-12,
            )
        )
        tree_best_index = int(np.nanargmax(tree_f1_curve))
    tree_operating_threshold = float(tree_thresholds[tree_best_index])

    tree_test_probability = tree_pipeline.predict_proba(X_test)[:, 1]
    tree_test_prediction = (
        tree_test_probability >= tree_operating_threshold
    ).astype(int)
    tree_top_k = max(1, int(math.ceil(0.05 * len(tree_test_probability))))
    tree_top_indices = np.argsort(-tree_test_probability)[:tree_top_k]
    tree_recall_at_top_5pct = float(
        y_test.iloc[tree_top_indices].sum() / max(int(y_test.sum()), 1)
    )
    tree_false_positives = int(
        ((tree_test_prediction == 1) & (y_test.to_numpy() == 0)).sum()
    )

    tree_metrics = {
        "model": "RANDOM_FOREST",
        "model_mode": MODEL_MODE,
        "distance_threshold_m": DISTANCE_THRESHOLD_M,
        "operating_threshold": tree_operating_threshold,
        "target_precision_requested": TARGET_PRECISION,
        "test_rows_sample": int(len(y_test)),
        "test_problem_rows_sample": int(y_test.sum()),
        "test_pr_auc_weighted": float(average_precision_score(
            y_test, tree_test_probability, sample_weight=w_test
        )),
        "test_roc_auc_weighted": float(roc_auc_score(
            y_test, tree_test_probability, sample_weight=w_test
        )),
        "test_precision_weighted": float(precision_score(
            y_test,
            tree_test_prediction,
            sample_weight=w_test,
            zero_division=0,
        )),
        "test_recall_weighted": float(recall_score(
            y_test,
            tree_test_prediction,
            sample_weight=w_test,
            zero_division=0,
        )),
        "test_f1_weighted": float(f1_score(
            y_test,
            tree_test_prediction,
            sample_weight=w_test,
            zero_division=0,
        )),
        "sample_false_alarms_per_1000": (
            1000.0 * tree_false_positives / max(len(y_test), 1)
        ),
        "sample_recall_at_top_5_percent": tree_recall_at_top_5pct,
        "tree_n_estimators": TREE_N_ESTIMATORS,
        "tree_max_depth": TREE_MAX_DEPTH,
        "tree_min_samples_leaf": TREE_MIN_SAMPLES_LEAF,
        "tree_include_text": TREE_INCLUDE_TEXT,
    }

    # Deterministic bounded sample from the untouched chronological test block.
    permutation_n = min(PERMUTATION_IMPORTANCE_MAX_ROWS, len(X_test))
    permutation_rng = np.random.default_rng(RANDOM_SEED)
    permutation_positions = np.sort(
        permutation_rng.choice(len(X_test), size=permutation_n, replace=False)
    )
    X_permutation = X_test.iloc[permutation_positions].copy()
    y_permutation = y_test.iloc[permutation_positions].copy()
    w_permutation = w_test.iloc[permutation_positions].copy()
    if y_permutation.nunique() < 2:
        raise RuntimeError(
            "Permutation-importance sample has only one target class. "
            "Increase PERMUTATION_IMPORTANCE_MAX_ROWS or extend ANALYSIS_DAYS."
        )

    permutation_kwargs = dict(
        estimator=tree_pipeline,
        X=X_permutation,
        y=y_permutation,
        scoring="average_precision",
        n_repeats=PERMUTATION_IMPORTANCE_REPEATS,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    try:
        permutation_result = permutation_importance(
            **permutation_kwargs,
            sample_weight=w_permutation,
        )
    except TypeError:
        # Compatibility with older scikit-learn releases.
        permutation_result = permutation_importance(**permutation_kwargs)

    tree_permutation_importance = pd.DataFrame({
        "MODEL_INPUT_FEATURE": X_permutation.columns,
        "PERMUTATION_IMPORTANCE_MEAN_PR_AUC_DROP": permutation_result.importances_mean,
        "PERMUTATION_IMPORTANCE_STD": permutation_result.importances_std,
    })

    def permutation_base_feature(model_input_feature):
        if model_input_feature == "__MODEL_TEXT":
            return "COMBINED_TECHNICAL_TEXT"
        if str(model_input_feature).startswith("MISSING__"):
            return str(model_input_feature).replace("MISSING__", "", 1)
        return str(model_input_feature)

    def permutation_source(model_input_feature):
        base_feature = permutation_base_feature(model_input_feature)
        if base_feature == "COMBINED_TECHNICAL_TEXT":
            return "MULTI_SOURCE_TEXT"
        return source_and_original(base_feature)[0]

    def permutation_feature_type(model_input_feature):
        if model_input_feature == "__MODEL_TEXT":
            return "TEXT_BUNDLE"
        if str(model_input_feature).startswith("MISSING__"):
            return "MISSINGNESS"
        return type_lookup.get(str(model_input_feature), "UNKNOWN")

    tree_permutation_importance.insert(
        1,
        "BASE_FEATURE",
        tree_permutation_importance["MODEL_INPUT_FEATURE"].map(
            permutation_base_feature
        ),
    )
    tree_permutation_importance.insert(
        2,
        "SOURCE_TABLE",
        tree_permutation_importance["MODEL_INPUT_FEATURE"].map(
            permutation_source
        ),
    )
    tree_permutation_importance.insert(
        3,
        "FEATURE_TYPE",
        tree_permutation_importance["MODEL_INPUT_FEATURE"].map(
            permutation_feature_type
        ),
    )
    tree_permutation_importance["STABILITY_LOWER_1SD"] = (
        tree_permutation_importance["PERMUTATION_IMPORTANCE_MEAN_PR_AUC_DROP"]
        - tree_permutation_importance["PERMUTATION_IMPORTANCE_STD"]
    )
    association_score_lookup = (
        active_shortlist.drop_duplicates("FEATURE")
        .set_index("FEATURE")["ASSOCIATION_SCORE"]
        .to_dict()
    )
    tree_permutation_importance["UNIVARIATE_ASSOCIATION_SCORE"] = (
        tree_permutation_importance["BASE_FEATURE"].map(
            association_score_lookup
        )
    )
    tree_permutation_importance = tree_permutation_importance.sort_values(
        [
            "PERMUTATION_IMPORTANCE_MEAN_PR_AUC_DROP",
            "STABILITY_LOWER_1SD",
        ],
        ascending=False,
    ).reset_index(drop=True)
    tree_permutation_importance.insert(
        0,
        "TREE_IMPACT_RANK",
        np.arange(1, len(tree_permutation_importance) + 1),
    )

    tree_feature_shortlist = tree_permutation_importance[
        tree_permutation_importance[
            "PERMUTATION_IMPORTANCE_MEAN_PR_AUC_DROP"
        ].gt(TREE_PERMUTATION_MIN_MEAN)
    ].head(TREE_TOP_FEATURES).copy()

    # Secondary diagnostic only: transformed one-hot/token impurity importance.
    transformed_feature_names = (
        tree_pipeline.named_steps["features"].get_feature_names_out()
    )
    impurity_values = tree_pipeline.named_steps["model"].feature_importances_
    tree_impurity_importance = pd.DataFrame({
        "TRANSFORMED_FEATURE": transformed_feature_names,
        "IMPURITY_IMPORTANCE_DIAGNOSTIC_ONLY": impurity_values,
    }).sort_values(
        "IMPURITY_IMPORTANCE_DIAGNOSTIC_ONLY",
        ascending=False,
    ).reset_index(drop=True)
    tree_impurity_importance.insert(
        0,
        "IMPURITY_RANK",
        np.arange(1, len(tree_impurity_importance) + 1),
    )

    tree_model_path = OUTPUT_DIR / f"random_forest_{MODEL_MODE.lower()}.joblib"
    tree_metrics_path = OUTPUT_DIR / f"random_forest_{MODEL_MODE.lower()}_metrics.json"
    dump(tree_pipeline, tree_model_path)
    tree_metrics_path.write_text(
        json.dumps(tree_metrics, indent=2),
        encoding="utf-8",
    )
    save_csv(
        tree_permutation_importance,
        "random_forest_permutation_importance.csv",
    )
    save_csv(
        tree_feature_shortlist,
        "random_forest_feature_shortlist.csv",
    )
    save_csv(
        tree_impurity_importance.head(500),
        "random_forest_impurity_importance_diagnostic.csv",
    )

    logistic_comparison = {
        "MODEL": "LOGISTIC_REGRESSION",
        **{
            key.upper(): value
            for key, value in baseline_metrics.items()
            if not isinstance(value, list)
        },
    }
    tree_comparison = {
        "MODEL": "RANDOM_FOREST",
        **{
            key.upper(): value
            for key, value in tree_metrics.items()
            if not isinstance(value, list) and key != "model"
        },
    }
    model_comparison = pd.DataFrame([
        logistic_comparison,
        tree_comparison,
    ])
    save_csv(model_comparison, "model_comparison_logistic_vs_random_forest.csv")

    display(Markdown("### Random Forest held-out metrics"))
    display(pd.DataFrame([
        {"METRIC": key, "VALUE": value}
        for key, value in tree_metrics.items()
    ]))
    display(Markdown("### Model-based feature impact — permutation importance"))
    display(tree_feature_shortlist.head(50))
    display(Markdown("### Logistic versus Random Forest"))
    display(model_comparison)

    if not tree_feature_shortlist.empty:
        import matplotlib.pyplot as plt

        tree_plot = tree_feature_shortlist.head(30).sort_values(
            "PERMUTATION_IMPORTANCE_MEAN_PR_AUC_DROP"
        )
        plt.figure(figsize=(12, max(5, 0.35 * len(tree_plot))))
        plt.barh(
            tree_plot["MODEL_INPUT_FEATURE"],
            tree_plot["PERMUTATION_IMPORTANCE_MEAN_PR_AUC_DROP"],
            xerr=tree_plot["PERMUTATION_IMPORTANCE_STD"],
            color="#C84C09",
            alpha=0.85,
        )
        plt.xlabel("Held-out PR-AUC decrease after shuffling")
        plt.title("Random Forest model-based feature impact")
        plt.grid(axis="x", alpha=0.25)
        plt.tight_layout()
        tree_plot_path = OUTPUT_DIR / "random_forest_permutation_importance.png"
        plt.savefig(tree_plot_path, dpi=160, bbox_inches="tight")
        plt.show()
        print("Saved:", tree_plot_path)
    print("Saved tree model:", tree_model_path)
else:
    print("RUN_TREE_MODEL=False; Random Forest confirmation was skipped.")


In [ ]:
# Output inventory and compact interpretation reminder.
output_files = sorted(
    path for path in OUTPUT_DIR.iterdir()
    if path.is_file()
)
output_inventory = pd.DataFrame([
    {"FILE": path.name, "SIZE_MB": round(path.stat().st_size / 1024**2, 3), "PATH": str(path)}
    for path in output_files
])
display(output_inventory)

display(Markdown(
    "### Interpretation\n"
    "- These are feature-target associations, not proof of causation.\n"
    "- Confirm the source definition of `DISTCELL2XY_M`; until then, the target is a large cell/location-gap proxy.\n"
    "- Use the pre-route shortlist for prediction. SIP/IMS/RAW outcomes belong in post-call detection/diagnosis.\n"
    "- Use `random_forest_feature_shortlist.csv` for multivariate model-based impact; permutation importance is primary and impurity importance is diagnostic only.\n"
    "- A strong ECGI/cell/USID result can be operationally useful but may memorize known bad infrastructure; validate on later dates and held-out cells.\n"
    "- Do not report sampled class prevalence as the population prevalence; use `target_summary.csv`."
))

con.close()
print("Complete. Outputs:", OUTPUT_DIR)
